<a href="https://colab.research.google.com/github/xiaoyuguo-del/TFM_Seguridad_Vial_Madrid/blob/main/04_Prediccion_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Librerías, carga de modelo y datos
# ══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/TFM_Seguridad_Vial'
outputs = f'{BASE}/outputs'
carpeta_bici    = f'{BASE}/datos/accidentes_bici/raw/'
carpeta_general = f'{BASE}/datos/accidentes_general/raw/'

# Cargamos modelo y scaler ya entrenados
ridge_final  = joblib.load(f'{outputs}/ridge_final.pkl')
scaler_full  = joblib.load(f'{outputs}/scaler_full.pkl')

# Cargamos serie base 2010-2024
serie = pd.read_csv(f'{outputs}/serie_mensual_2010_2024_corregida.csv')
serie['fecha'] = pd.to_datetime(
    serie['año'].astype(str) + '-' + serie['mes'].astype(str).str.zfill(2) + '-01'
)

# Feature cols — mismo orden que en entrenamiento
distritos_serie = sorted(serie['distrito'].unique())
feature_cols = (
    ['lag_1', 'lag_12', 'tendencia_3m', 'cambio_estructura', 'covid_imputado'] +
    [f'mes_{m}' for m in range(2, 13)] +
    [f'dist_{re.sub("[^a-zA-Z0-9]", "_", d)}' for d in distritos_serie]
)

print(f'✅ Modelo cargado')
print(f'✅ Serie base: {serie.shape} | Años: {sorted(serie["año"].unique())}')
print(f'✅ Features: {len(feature_cols)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Modelo cargado
✅ Serie base: (3780, 20) | Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
✅ Features: 37


In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 2 — Construir serie mensual real de 2025
# ══════════════════════════════════════════════════════════

# Misma lógica que el ETL principal pero solo para 2025

mapa_gravedad = {
    1: 'leve', 2: 'leve', 5: 'leve', 6: 'leve', 7: 'leve',
    3: 'grave', 4: 'fallecido',
    14: 'sin_asistencia', 77: 'desconocido'
}
jerarquia_gravedad  = {'fallecido': 4, 'grave': 3, 'leve': 2,
                       'sin_asistencia': 1, 'desconocido': 0}
jerarquia_inversa   = {v: k for k, v in jerarquia_gravedad.items()}
orden_gravedad      = jerarquia_gravedad
rank_a_gravedad     = jerarquia_inversa

# Cargamos CSVs raw de 2025
bici_2025 = pd.read_csv(f'{carpeta_bici}AccidentesBici2025.csv',
                         sep=';', encoding='utf-8-sig')
bici_2025 = bici_2025.rename(columns={'tipo_vehículo': 'tipo_vehiculo'})
bici_2025 = bici_2025.loc[:, ~bici_2025.columns.duplicated()]
bici_2025['fecha_dt'] = pd.to_datetime(bici_2025['fecha'], dayfirst=True)
bici_2025['año'] = bici_2025['fecha_dt'].dt.year
bici_2025['mes'] = bici_2025['fecha_dt'].dt.month
bici_2025['gravedad'] = bici_2025['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')

general_2025 = pd.read_csv(f'{carpeta_general}Accidentes2025.csv',
                            sep=';', encoding='utf-8-sig')
general_2025 = general_2025.rename(columns={'tipo_vehículo': 'tipo_vehiculo'})
general_2025 = general_2025.loc[:, ~general_2025.columns.duplicated()]
general_2025['fecha_dt'] = pd.to_datetime(general_2025['fecha'], dayfirst=True)
general_2025['año'] = general_2025['fecha_dt'].dt.year
general_2025['mes'] = general_2025['fecha_dt'].dt.month

peatones_2025 = general_2025[general_2025['tipo_persona'] == 'Peatón'].copy()
peatones_2025['gravedad'] = peatones_2025['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')

# Deduplicación VRU — mismo criterio que ETL
vru_2025 = pd.concat([
    bici_2025.assign(origen='bici'),
    peatones_2025.assign(origen='peaton')
], ignore_index=True)
vru_2025['gravedad_rank'] = vru_2025['gravedad'].map(orden_gravedad)

vru_unicos_2025 = (
    vru_2025.groupby(['distrito', 'año', 'mes', 'num_expediente'], as_index=False)
            .agg(gravedad_rank=('gravedad_rank', 'max'))
)
vru_unicos_2025['gravedad_max'] = vru_unicos_2025['gravedad_rank'].map(rank_a_gravedad)

serie_2025 = (
    vru_unicos_2025
    .groupby(['distrito', 'año', 'mes'])
    .agg(
        acc_vru_total   =('num_expediente', 'nunique'),
        acc_vru_leves   =('gravedad_max', lambda x: (x == 'leve').sum()),
        acc_vru_graves  =('gravedad_max', lambda x: (x == 'grave').sum()),
        acc_vru_mortales=('gravedad_max', lambda x: (x == 'fallecido').sum())
    )
    .reset_index()
)
serie_2025['acc_ponderado'] = (
    serie_2025['acc_vru_leves']    * 1 +
    serie_2025['acc_vru_graves']   * 7 +
    serie_2025['acc_vru_mortales'] * 16
)
serie_2025['cambio_estructura'] = 1
serie_2025['covid_imputado']    = 0
serie_2025['fecha'] = pd.to_datetime(
    serie_2025['año'].astype(str) + '-' +
    serie_2025['mes'].astype(str).str.zfill(2) + '-01'
)

print(f'✅ Serie 2025 construida: {serie_2025.shape}')
print(f'Meses disponibles: {sorted(serie_2025["mes"].unique())}')
print(f'Distritos: {serie_2025["distrito"].nunique()}')
print(f'\nacc_ponderado 2025 por distrito:')
print(serie_2025.groupby('distrito')['acc_ponderado'].sum().sort_values(ascending=False).to_string())

✅ Serie 2025 construida: (251, 11)
Meses disponibles: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Distritos: 21

acc_ponderado 2025 por distrito:
distrito
SALAMANCA              248
CENTRO                 245
CARABANCHEL            239
MONCLOA-ARAVACA        231
LATINA                 213
CHAMBERÍ               202
FUENCARRAL-EL PARDO    200
ARGANZUELA             199
PUENTE DE VALLECAS     189
CIUDAD LINEAL          181
TETUÁN                 171
VILLAVERDE             168
CHAMARTÍN              160
USERA                  150
RETIRO                 140
HORTALEZA              137
SAN BLAS-CANILLEJAS    127
VILLA DE VALLECAS       89
BARAJAS                 77
MORATALAZ               60
VICÁLVARO               44


In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 3 — Predicción recursiva 2026
# ══════════════════════════════════════════════════════════

predicciones_2026 = []

for distrito in sorted(distritos_serie):

    # Historial real: 2010-2024 + 2025 real
    d_hist = serie[serie['distrito'] == distrito].sort_values('fecha')
    d_2025 = serie_2025[serie_2025['distrito'] == distrito].sort_values('mes')

    # Construimos historial completo con datos reales
    pred_history = dict(zip(
        zip(d_hist['año'], d_hist['mes']),
        d_hist['acc_ponderado']
    ))
    # Añadimos 2025 real
    for _, row in d_2025.iterrows():
        pred_history[(2025, int(row['mes']))] = row['acc_ponderado']

    for mes in range(1, 13):

        # lag_1: mes anterior real o predicho
        if mes == 1:
            lag_1 = pred_history.get((2025, 12), 0)
        else:
            lag_1 = pred_history.get((2026, mes - 1), 0)

        # lag_12: mismo mes de 2025 — siempre real
        lag_12 = pred_history.get((2025, mes), 0)

        # tendencia_3m: 3 meses anteriores reales/predichos
        if mes == 1:
            ultimos = [pred_history.get((2025, 10), 0),
                       pred_history.get((2025, 11), 0),
                       pred_history.get((2025, 12), 0)]
        elif mes == 2:
            ultimos = [pred_history.get((2025, 11), 0),
                       pred_history.get((2025, 12), 0),
                       pred_history.get((2026,  1), 0)]
        elif mes == 3:
            ultimos = [pred_history.get((2025, 12), 0),
                       pred_history.get((2026,  1), 0),
                       pred_history.get((2026,  2), 0)]
        else:
            ultimos = [pred_history.get((2026, mes - 3), 0),
                       pred_history.get((2026, mes - 2), 0),
                       pred_history.get((2026, mes - 1), 0)]

        tend = np.polyfit([0, 1, 2], ultimos, 1)[0]

        mes_dummies  = {f'mes_{m}': int(mes == m) for m in range(1, 13)}
        dist_dummies = {
            f'dist_{re.sub("[^a-zA-Z0-9]", "_", d2)}': int(d2 == distrito)
            for d2 in distritos_serie
        }

        fila = {
            'distrito':          distrito,
            'año':               2026,
            'mes':               mes,
            'lag_1':             lag_1,
            'lag_12':            lag_12,
            'tendencia_3m':      tend,
            'cambio_estructura': 1,
            'covid_imputado':    0,
            **mes_dummies,
            **dist_dummies
        }

        X_mes    = scaler_full.transform(
            pd.DataFrame([fila])[feature_cols].values
        )
        pred_mes = float(np.maximum(ridge_final.predict(X_mes), 0))

        pred_history[(2026, mes)] = pred_mes
        fila['acc_pred_mensual'] = pred_mes
        predicciones_2026.append(fila)

df_2026 = pd.DataFrame(predicciones_2026)

# Verificación recursividad
ene = df_2026[(df_2026['distrito'] == 'CENTRO') & (df_2026['mes'] == 1)]['acc_pred_mensual'].values[0]
feb_lag = df_2026[(df_2026['distrito'] == 'CENTRO') & (df_2026['mes'] == 2)]['lag_1'].values[0]
assert abs(ene - feb_lag) < 0.01, "⚠️ Recursividad incorrecta"
print(f'✅ Recursividad verificada — lag_1(feb) = pred(ene) = {feb_lag:.2f}')

# Suma anual → IRP 2026
irp_2026 = (
    df_2026.groupby('distrito')['acc_pred_mensual']
    .sum()
    .reset_index()
    .rename(columns={'acc_pred_mensual': 'acc_pred_2026'})
)
irp_2026['acc_pred_2026'] = irp_2026['acc_pred_2026'].round(2)

mn = irp_2026['acc_pred_2026'].min()
mx = irp_2026['acc_pred_2026'].max()
irp_2026['IRP_2026'] = ((irp_2026['acc_pred_2026'] - mn) / (mx - mn) * 100).round(2)

irp_2026 = irp_2026.sort_values('IRP_2026', ascending=False).reset_index(drop=True)
irp_2026.index += 1

print('\n=== RANKING IRP — PREDICCIÓN 2026 ===')
print(irp_2026[['distrito', 'acc_pred_2026', 'IRP_2026']].to_string())

irp_2026.to_csv(f'{outputs}/IRP_prediccion_2026.csv',
                index=True, index_label='posicion',
                encoding='utf-8-sig')
print('\n✅ IRP_prediccion_2026.csv guardado')

✅ Recursividad verificada — lag_1(feb) = pred(ene) = 22.06

=== RANKING IRP — PREDICCIÓN 2026 ===
               distrito  acc_pred_2026  IRP_2026
1                CENTRO         287.31    100.00
2    PUENTE DE VALLECAS         233.74     77.24
3           CARABANCHEL         216.07     69.74
4   FUENCARRAL-EL PARDO         203.59     64.44
5             SALAMANCA         203.05     64.21
6                LATINA         201.88     63.71
7              CHAMBERÍ         189.63     58.51
8         CIUDAD LINEAL         181.12     54.89
9             CHAMARTÍN         171.57     50.84
10      MONCLOA-ARAVACA         169.14     49.80
11               TETUÁN         167.45     49.09
12  SAN BLAS-CANILLEJAS         161.25     46.45
13           ARGANZUELA         157.94     45.05
14               RETIRO         150.82     42.02
15            HORTALEZA         134.73     35.19
16                USERA         134.69     35.17
17           VILLAVERDE         115.11     26.85
18    VILLA DE VALLE

In [ ]:
# Cargar datos de entrenamiento guardados
import joblib
X_full = joblib.load(f'{outputs}/X_full.pkl')
y_full = joblib.load(f'{outputs}/y_full.pkl')
feature_cols = joblib.load(f'{outputs}/feature_cols.pkl')
print(f'✅ X_full: {X_full.shape}')
print(f'✅ y_full: {y_full.shape}')
print(f'✅ feature_cols: {len(feature_cols)} features')

✅ X_full: (3528, 37)
✅ y_full: (3528,)
✅ feature_cols: 37 features


In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA BOOTSTRAP — Intervalos de confianza IRP 2026
# ══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import re

N_BOOT = 500
np.random.seed(42)

print(f'Ejecutando {N_BOOT} iteraciones de bootstrapping...')
print('Esto puede tardar ~30 minutos en Colab.\n')

boot_anuales = {d: [] for d in distritos_serie}

for i in range(N_BOOT):
    if i % 50 == 0:
        print(f'  Iteración {i}/{N_BOOT}...')

    # Resamplear con reemplazo
    X_boot, y_boot = resample(X_full, y_full, random_state=i)

    # Reentrenar modelo
    scaler_b = StandardScaler()
    X_boot_scaled = scaler_b.fit_transform(X_boot)
    ridge_b = Ridge(alpha=500)
    ridge_b.fit(X_boot_scaled, y_boot)

    # Predicción recursiva 2026
    for distrito in sorted(distritos_serie):
        d_hist = serie[serie['distrito'] == distrito].sort_values('fecha')
        d_2025 = serie_2025[serie_2025['distrito'] == distrito].sort_values('mes')

        pred_history = dict(zip(
            zip(d_hist['año'], d_hist['mes']),
            d_hist['acc_ponderado']
        ))
        for _, row in d_2025.iterrows():
            pred_history[(2025, int(row['mes']))] = row['acc_ponderado']

        total_anual = 0
        for mes in range(1, 13):
            if mes == 1:
                lag_1 = pred_history.get((2025, 12), 0)
            else:
                lag_1 = pred_history.get((2026, mes - 1), 0)

            lag_12 = pred_history.get((2025, mes), 0)

            if mes == 1:
                ultimos = [pred_history.get((2025, 10), 0),
                           pred_history.get((2025, 11), 0),
                           pred_history.get((2025, 12), 0)]
            elif mes == 2:
                ultimos = [pred_history.get((2025, 11), 0),
                           pred_history.get((2025, 12), 0),
                           pred_history.get((2026, 1), 0)]
            elif mes == 3:
                ultimos = [pred_history.get((2025, 12), 0),
                           pred_history.get((2026, 1), 0),
                           pred_history.get((2026, 2), 0)]
            else:
                ultimos = [pred_history.get((2026, mes - 3), 0),
                           pred_history.get((2026, mes - 2), 0),
                           pred_history.get((2026, mes - 1), 0)]

            tend = np.polyfit([0, 1, 2], ultimos, 1)[0]

            mes_dummies = {f'mes_{m}': int(mes == m) for m in range(1, 13)}
            dist_dummies = {
                f'dist_{re.sub("[^a-zA-Z0-9]", "_", d2)}': int(d2 == distrito)
                for d2 in distritos_serie
            }

            fila = {
                'distrito':          distrito,
                'año':               2026,
                'mes':               mes,
                'lag_1':             lag_1,
                'lag_12':            lag_12,
                'tendencia_3m':      tend,
                'cambio_estructura': 1,
                'covid_imputado':    0,
                **mes_dummies,
                **dist_dummies
            }

            X_mes = scaler_b.transform(
                pd.DataFrame([fila])[feature_cols].values
            )
            pred_mes = float(np.maximum(ridge_b.predict(X_mes), 0))
            pred_history[(2026, mes)] = pred_mes
            total_anual += pred_mes

        boot_anuales[distrito].append(total_anual)

# Calcular IC 95%
print('\n=== INTERVALOS DE CONFIANZA 95% — IRP 2026 ===\n')

resultados = []
for distrito in sorted(distritos_serie):
    preds = np.array(boot_anuales[distrito])
    pred_central = irp_2026[irp_2026['distrito'] == distrito]['acc_pred_2026'].values[0]
    resultados.append({
        'distrito': distrito,
        'pred_central': pred_central,
        'ic_lower': round(np.percentile(preds, 2.5), 1),
        'ic_upper': round(np.percentile(preds, 97.5), 1)
    })

df_ic = (pd.DataFrame(resultados)
         .sort_values('pred_central', ascending=False)
         .reset_index(drop=True))
df_ic.index += 1
df_ic.index.name = 'posicion'

print(df_ic.to_string())

df_ic.to_csv(f'{outputs}/IRP_2026_bootstrap_IC.csv',
             index=True, encoding='utf-8-sig')
print('\n✅ IRP_2026_bootstrap_IC.csv guardado')

Ejecutando 500 iteraciones de bootstrapping...
Esto puede tardar ~30 minutos en Colab.

  Iteración 0/500...
  Iteración 50/500...
  Iteración 100/500...
  Iteración 150/500...
  Iteración 200/500...
  Iteración 250/500...
  Iteración 300/500...
  Iteración 350/500...
  Iteración 400/500...
  Iteración 450/500...

=== INTERVALOS DE CONFIANZA 95% — IRP 2026 ===

                     distrito  pred_central  ic_lower  ic_upper
posicion                                                       
1                      CENTRO        287.31     267.4     307.3
2          PUENTE DE VALLECAS        233.74     216.4     251.6
3                 CARABANCHEL        216.07     199.8     235.3
4         FUENCARRAL-EL PARDO        203.59     188.4     217.6
5                   SALAMANCA        203.05     187.0     219.3
6                      LATINA        201.88     187.2     218.3
7                    CHAMBERÍ        189.63     173.7     207.9
8               CIUDAD LINEAL        181.12     164.2     19

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4 — Score final con IRP 2026
# ══════════════════════════════════════════════════════════

# Cargamos IPD ranking medio 2021-2024 (no cambia)
ipd_ranking = pd.read_csv(f'{outputs}/IPD_ranking_medio.csv')
ipd_ranking.columns = ['posicion', 'distrito', 'IPD_medio']

mn_ipd = ipd_ranking['IPD_medio'].min()
mx_ipd = ipd_ranking['IPD_medio'].max()
ipd_ranking['IPD_norm'] = (
    (ipd_ranking['IPD_medio'] - mn_ipd) / (mx_ipd - mn_ipd) * 100
).round(2)

# Score final con IRP 2026
score_2026 = ipd_ranking[['distrito', 'IPD_medio', 'IPD_norm']].merge(
    irp_2026[['distrito', 'acc_pred_2026', 'IRP_2026']],
    on='distrito', how='inner'
)
score_2026['Score_final_2026'] = (
    score_2026['IPD_norm']  * 0.60 +
    score_2026['IRP_2026']  * 0.40
).round(2)

score_2026 = score_2026.sort_values('Score_final_2026', ascending=False).reset_index(drop=True)
score_2026.index += 1

print('=== RANKING FINAL 2026 — SCORE = 0.60×IPD + 0.40×IRP ===')
print(f'{"Pos":<5} {"Distrito":<25} {"IPD_norm":>10} {"IRP_2026":>10} {"Score":>8}')
print('-' * 62)
for pos, row in score_2026.iterrows():
    print(f'{pos:<5} {row["distrito"]:<25} {row["IPD_norm"]:>10.2f} {row["IRP_2026"]:>10.2f} {row["Score_final_2026"]:>8.2f}')

# Comparación con ranking 2025
score_2025 = pd.read_csv(f'{outputs}/score_final.csv')
score_2025.columns = ['posicion', 'distrito', 'IPD_medio', 'IPD_norm', 'IRP_2025', 'Score_2025']

comp = score_2026[['distrito', 'Score_final_2026']].merge(
    score_2025[['distrito', 'Score_2025']],
    on='distrito'
)
comp['rank_2026'] = comp['Score_final_2026'].rank(ascending=False).astype(int)
comp['rank_2025'] = comp['Score_2025'].rank(ascending=False).astype(int)
comp['cambio']    = comp['rank_2025'] - comp['rank_2026']

from scipy.stats import spearmanr
rho, p = spearmanr(comp['rank_2025'], comp['rank_2026'])

print(f'\n=== ESTABILIDAD DEL RANKING 2025 → 2026 ===')
print(f'Correlación Spearman: rho={rho:.4f} (p={p:.4f})')
print(f'\n{"Distrito":<25} {"Rank 2025":>10} {"Rank 2026":>10} {"Cambio":>8}')
print('-' * 55)
for _, row in comp.sort_values('rank_2026').iterrows():
    flecha = '↑' if row['cambio'] > 0 else '↓' if row['cambio'] < 0 else '→'
    print(f'{row["distrito"]:<25} {row["rank_2025"]:>10} {row["rank_2026"]:>10} {flecha} {abs(int(row["cambio"]))}')

score_2026.to_csv(f'{outputs}/score_final_2026.csv',
                  index=True, index_label='posicion',
                  encoding='utf-8-sig')
print('\n✅ score_final_2026.csv guardado')

=== RANKING FINAL 2026 — SCORE = 0.60×IPD + 0.40×IRP ===
Pos   Distrito                    IPD_norm   IRP_2026    Score
--------------------------------------------------------------
1     CENTRO                         82.20     100.00    89.32
2     CHAMBERÍ                      100.00      58.51    83.40
3     SALAMANCA                      87.55      64.21    78.21
4     RETIRO                         92.39      42.02    72.24
5     PUENTE DE VALLECAS             52.86      77.24    62.61
6     CIUDAD LINEAL                  66.02      54.89    61.57
7     TETUÁN                         69.09      49.09    61.09
8     CHAMARTÍN                      67.71      50.84    60.96
9     LATINA                         57.06      63.71    59.72
10    ARGANZUELA                     62.71      45.05    55.65
11    CARABANCHEL                    43.73      69.74    54.13
12    MONCLOA-ARAVACA                46.48      49.80    47.81
13    MORATALAZ                      73.95       7.88    47.5

In [ ]:
# ══════════════════════════════════════════════════════════
# TABLA COMPARATIVA IRP 2025 vs 2026
# ══════════════════════════════════════════════════════════

import pandas as pd
from scipy.stats import spearmanr

outputs = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs'

# Cargamos los dos rankings
irp_2025 = pd.read_csv(f'{outputs}/IRP_prediccion_2025.csv')
irp_2026 = pd.read_csv(f'{outputs}/IRP_prediccion_2026.csv')

# Renombramos columnas
irp_2025 = irp_2025.rename(columns={'IRP': 'IRP_2025', 'posicion': 'pos_2025'})
irp_2026 = irp_2026.rename(columns={'posicion': 'pos_2026'})

# Unimos y ordenamos por IRP_2026
tabla = irp_2025.merge(irp_2026, on='distrito', how='left')
tabla = (tabla[['distrito', 'acc_pred_2025', 'IRP_2025', 'acc_pred_2026', 'IRP_2026']]
         .sort_values('IRP_2026', ascending=False)
         .reset_index(drop=True))
tabla.index += 1
tabla.index.name = 'Pos. 2026'

# Reordenamos columnas para el TFM
tabla_tfm = (tabla
    .reset_index()
    [['Pos. 2026', 'distrito', 'acc_pred_2026', 'IRP_2026', 'acc_pred_2025', 'IRP_2025']])

print('=== TABLA COMPARATIVA IRP 2025 vs 2026 ===')
print(tabla_tfm.to_string(index=False))

# Correlación de Spearman entre rankings
rho, pval = spearmanr(irp_2025['pos_2025'], irp_2026['pos_2026'])
print(f'\nCorrelación de Spearman 2025 vs 2026: ρ = {rho:.4f} (p = {pval:.4f})')

=== TABLA COMPARATIVA IRP 2025 vs 2026 ===
 Pos. 2026            distrito  acc_pred_2026  IRP_2026  acc_pred_2025  IRP_2025
         1              CENTRO         287.31    100.00         294.32    100.00
         2  PUENTE DE VALLECAS         233.74     77.24         243.32     78.63
         3         CARABANCHEL         216.07     69.74         210.83     65.01
         4 FUENCARRAL-EL PARDO         203.59     64.44         197.68     59.50
         5           SALAMANCA         203.05     64.21         197.55     59.45
         6              LATINA         201.88     63.71         196.71     59.10
         7            CHAMBERÍ         189.63     58.51         189.74     56.17
         8       CIUDAD LINEAL         181.12     54.89         178.87     51.62
         9           CHAMARTÍN         171.57     50.84         174.92     49.96
        10     MONCLOA-ARAVACA         169.14     49.80         160.63     43.98
        11              TETUÁN         167.45     49.09         16

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

outputs = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs'
val = pd.read_csv(f'{outputs}/validacion_IRP_2025.csv')
rho, pval = spearmanr(val['rank_pred_2025'], val['rank_real_2025'])
print(f'Rho validación externa 2025: {rho:.3f} (p={pval:.4f})')
print(f'Error porcentual medio: {val["error_pct"].abs().mean():.1f}%')
print(f'Error porcentual mediano: {val["error_pct"].abs().median():.1f}%')
print(f'Distritos con diff <= 1: {(val["diff_rank"] <= 1).sum()}')

Rho validación externa 2025: 0.834 (p=0.0000)
Error porcentual medio: 15.4%
Error porcentual mediano: 11.8%
Distritos con diff <= 1: 11
